# Django Fundamentals — Comprehensive Learning Notebook

This notebook covers all 8 modules of the Django Fundamentals course. Each section builds on the previous one, culminating in a complete meeting-planning web application.

---

## Table of Contents

0. [Terminal Workflow Reference](#django-project-workflow--terminal-reference)
1. [Starting a Django Project](#1-starting-a-django-project)
2. [Creating a Simple Web Page](#2-creating-a-simple-web-page)
3. [Setting Up a Data Model](#3-setting-up-a-data-model)
4. [Combining Model, View and Template (MVT)](#4-combining-model-view-and-template-mvt)
5. [URLs and Link Building](#5-urls-and-link-building)
6. [Templates, Styling and Static Content](#6-templates-styling-and-static-content)
7. [Adding User Interaction with ModelForms](#7-adding-user-interaction-with-modelforms)
8. [Editing Meetings and Authentication](#8-editing-meetings-and-authentication)
9. [Quick-Reference Cheatsheet](#quick-reference-cheatsheet)

---
## Django Project Workflow — Terminal Reference

End-to-end sequence from a blank directory to a running app. Follow these phases in order.

---

### Phase 1 — Environment Setup

Always work inside a virtual environment. Do this once per project.

```bash
python -m venv .venv
source .venv/bin/activate      # macOS / Linux
# .venv\Scripts\activate       # Windows

pip install django
```

---

### Phase 2 — Bootstrap the Project

```bash
django-admin startproject meeting_planner .   # the trailing dot avoids a nested directory
python manage.py runserver                    # smoke-test: http://127.0.0.1:8000/
```

> Use underscores in project and app names — hyphens are not valid Python identifiers.  
> The trailing `.` creates `manage.py` in the current directory instead of inside a subdirectory.

---

### Phase 3 — Initial Migration

Django ships with several built-in apps (auth, sessions, admin). Apply their migrations before doing anything else.

```bash
python manage.py showmigrations   # see what is pending
python manage.py migrate          # create auth/session/admin tables
python manage.py dbshell          # verify
```

```sql
.tables                           -- should list django_*, auth_* tables
SELECT * FROM django_migrations;  -- all applied migrations
.exit
```

---

### Phase 4 — Create Apps

```bash
python manage.py startapp website
python manage.py startapp meetings
```

Then register both in `settings.py`:

```python
# meeting_planner/settings.py
INSTALLED_APPS = [
    ...
    'website',
    'meetings',
]
```

---

### Phase 5 — Define Models & Migrate

After writing your models in `meetings/models.py`:

```bash
python manage.py makemigrations           # generate migration scripts
python manage.py sqlmigrate meetings 0001 # optional: preview the SQL
python manage.py migrate                  # apply to the database
python manage.py showmigrations           # confirm all boxes are checked
```

Verify the new tables:

```bash
python manage.py dbshell
```

```sql
.tables
SELECT * FROM meetings_meeting;
.exit
```

---

### Phase 6 — Admin & Superuser

```bash
python manage.py createsuperuser
# → prompts for username, email, password
# → visit http://127.0.0.1:8000/admin/
```

---

### Quick Reference — Full Sequence

```bash
# 1. Setup
python -m venv .venv && source .venv/bin/activate
pip install django

# 2. Project
django-admin startproject meeting_planner .
python manage.py migrate
python manage.py runserver

# 3. Apps
python manage.py startapp website
python manage.py startapp meetings
# → add to INSTALLED_APPS, write models

# 4. App migrations
python manage.py makemigrations
python manage.py migrate

# 5. Admin access
python manage.py createsuperuser
python manage.py runserver
```

### Key `manage.py` Commands

| Command | Purpose |
|---|---|
| `runserver` | Start dev server (auto-reloads on code changes) |
| `startapp <name>` | Scaffold a new app |
| `makemigrations` | Generate migration files from model changes |
| `showmigrations` | List all migrations and their applied status |
| `sqlmigrate <app> <num>` | Preview SQL for a migration without running it |
| `migrate` | Apply pending migrations |
| `dbshell` | Open an interactive shell for the configured database |
| `createsuperuser` | Create an admin account |
| `shell` | Interactive Python shell with Django fully loaded |

---
## 1. Starting a Django Project

### Version Requirements

- Django 6.x  
- Python 3.10 or later

### Installation & Setup

**Best practice:** always install packages inside a **virtual environment**.  
PyCharm (and most IDEs) create one automatically when you start a project.

```bash
# Run these in your terminal (not in the notebook)

# 1. Create and activate a virtual environment
python -m venv .venv
source .venv/bin/activate        # macOS/Linux
# .venv\Scripts\activate         # Windows

# 2. Install Django
pip install django

# 3. Create a new project (trailing dot keeps manage.py at top level)
django-admin startproject meeting_planner .

# 4. Start the development server
python manage.py runserver       # → http://127.0.0.1:8000/
```

### Key Facts About the Development Server

| Fact | Detail |
|---|---|
| Auto-reloads | Picks up Python code changes without restart |
| Stop it | `Ctrl+C` |
| **Never use in production** | Use a proper WSGI/ASGI server (Gunicorn, uvicorn, etc.) instead |

### Exploring the Generated Project Structure

After `django-admin startproject meeting_planner .` you get:

```
./                            ← your working directory
├── manage.py                 ← management command runner
└── meeting_planner/          ← Python package (project config)
    ├── __init__.py
    ├── settings.py           ← all project configuration
    ├── urls.py               ← root URL router
    ├── asgi.py               ← ASGI entry point (async servers)
    └── wsgi.py               ← WSGI entry point (traditional servers)
```

### Key `settings.py` Options

```python
# meeting_planner/settings.py — important settings to know

DEBUG = True          # Set to False in production — never expose tracebacks publicly

ALLOWED_HOSTS = []    # Add your domain/IP in production: ['example.com', '127.0.0.1']

DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.sqlite3',   # swap for postgresql, mysql, etc.
        'NAME': BASE_DIR / 'db.sqlite3',
    }
}

INSTALLED_APPS = [    # Register your apps here
    'django.contrib.admin',
    'django.contrib.auth',
    'django.contrib.contenttypes',
    'django.contrib.sessions',
    'django.contrib.messages',
    'django.contrib.staticfiles',
    # your apps:
    'website',
    'meetings',
]

STATIC_URL = 'static/'           # URL prefix for static files

LOGIN_REDIRECT_URL = '/'         # Where to go after successful login
LOGOUT_REDIRECT_URL = '/'        # Where to go after logout
LOGIN_URL = '/accounts/login/'   # Default — where @login_required redirects
```

### Summary — Module 1
- Installed Django in a virtual environment  
- Created a project with `django-admin startproject`  
- Ran it with `python manage.py runserver`  
- Explored the generated project files and key settings

---
## 2. Creating a Simple Web Page

### Django Apps

A Django **project** contains one or more **apps**.

| Concept | What it is |
|---|---|
| Project | The whole website configuration |
| App | A self-contained Python package with models, views, templates, urls |

**Guidelines:**
- Apps can be reused between projects  
- Keep apps small and simple, each with a clear purpose  
- Most Django projects contain several apps

```bash
# In your terminal, inside the project directory:
python manage.py startapp website
```

```python
# Then register the app in settings.py:
INSTALLED_APPS = [
    ...,
    'website',
]
```

### Adding Your First Page

Two files always work together to produce a page:

**`website/views.py`** — the view function  
**`meeting_planner/urls.py`** — the URL router

```python
# website/views.py
from django.http import HttpResponse

def welcome(request):
    return HttpResponse("Welcome!")
```

```python
# meeting_planner/urls.py
from django.contrib import admin
from django.urls import path
from website.views import welcome

urlpatterns = [
    path('admin/', admin.site.urls),
    path('', welcome),      # maps the root URL '/' to the welcome view
]
```

### How Django Serves a Page

```
Web Browser                              Server
┌─────────────────────┐                 ┌──────────────────────────────┐
│  127.0.0.1:8000/    │─── HTTP GET / ─►│  Find mapping in urls.py     │
│                     │                 │  path('', welcome)           │
│                     │                 │           │                  │
│  Welcome!           │◄────────────────│  def welcome(request):       │
└─────────────────────┘                 │    return HttpResponse("...")│
                                        └──────────────────────────────┘
```

**Key concept:** Control flow is dictated by incoming HTTP requests.  
A view function is **only called** when its URL is requested.

### Summary — Module 2
- Created a Django app with `startapp`  
- Wrote a view function that returns `HttpResponse`  
- Mapped a URL to the view in `urls.py`

---
## 3. Setting Up a Data Model

### Django Models

| Concept | Detail |
|---|---|
| Models | Python classes mapped to database tables |
| Fields | Mapped to columns |
| Objects | Each instance = one database row |
| SQL | Generated automatically by Django |
| Supported DBs | PostgreSQL, MariaDB, MySQL, Oracle, SQLite (+ more via packages) |

### Model Classes

```python
# meetings/models.py
from django.db import models

class Meeting(models.Model):
    title    = models.CharField(max_length=200)
    date     = models.DateField()
    start_time = models.TimeField()
    end_time   = models.TimeField()
    location = models.CharField(max_length=200)

    def __str__(self):
        return self.title


# Generic pattern (from the slides):
class Person(models.Model):
    name = models.CharField(max_length=100)
    age  = models.IntegerField()
```

### Why `__str__` Matters

`__str__` controls how an object is displayed everywhere Django renders it — the admin list, the shell, error messages, and form select widgets.

```python
# Without __str__:   Meeting object (1)
# With __str__:      Team Standup
```

### `ForeignKey` — `on_delete` Options

When a referenced object is deleted, Django needs to know what to do with dependent rows:

| Option | Behaviour |
|---|---|
| `CASCADE` | Delete the dependent rows too |
| `PROTECT` | Raise an error — prevents deletion of the referenced object |
| `SET_NULL` | Set the FK column to `NULL` (field must have `null=True`) |
| `SET_DEFAULT` | Set the FK column to its `default` value |
| `DO_NOTHING` | Leave the FK pointing at a non-existent row (dangerous) |

```python
# Example: deleting a User deletes all their meetings
organizer = models.ForeignKey(User, on_delete=models.CASCADE)

# Example: prevent deleting a Location that still has meetings assigned
location  = models.ForeignKey(Location, on_delete=models.PROTECT)
```

### Models vs Migrations

```
  Models                      Migrations
  ──────                      ──────────
  Python classes              Python scripts
  Mapped to DB tables         Keep DB structure in sync with code
  Each object = a row         Auto-generated (but not always)
```

### Migration Workflow

```bash
# Step 0: make sure your app is in INSTALLED_APPS in settings.py

# Step 1: change the model code (add/edit/remove fields)

# Step 2: generate a migration script
python manage.py makemigrations

# Optional: preview the SQL before applying
python manage.py sqlmigrate meetings 0001

# Step 3: apply migrations to the database
python manage.py migrate

# Confirm all migrations are applied
python manage.py showmigrations
```

### Using the Django Shell

`manage.py shell` opens an interactive Python session with the full Django environment loaded — models, settings, everything. Useful for testing queries before writing views.

```bash
python manage.py shell
```

```python
# Inside the shell:
from meetings.models import Meeting

Meeting.objects.all()           # → <QuerySet []>
Meeting.objects.count()         # → 0

# Create a record
m = Meeting(title="Standup", date="2026-10-15",
            start_time="09:00", end_time="09:15", location="Room A")
m.save()

# Or shorthand
Meeting.objects.create(title="Retro", date="2026-10-20",
                       start_time="14:00", end_time="15:00", location="Online")

# Query
Meeting.objects.filter(location="Online")
Meeting.objects.get(pk=1)

exit()
```

### Registering Models with the Admin Site

```python
# meetings/admin.py
from django.contrib import admin
from .models import Meeting

admin.site.register(Meeting)
```

```bash
python manage.py createsuperuser
# Then visit http://127.0.0.1:8000/admin/
```

### Summary — Module 4
- Defined model classes that inherit from `models.Model`  
- Ran `makemigrations` and `migrate` to create the database table  
- Registered models with the Admin site  
- Created a superuser and added data via the admin interface

---
## 4. Combining Model, View and Template (MVT)

### The Model-Template-View Pattern

Django calls its architecture **MTV** (Model–Template–View), equivalent to MVC (Model–View–Controller):

```
┌──────────────┐   ┌────────────────────┐   ┌──────────────────┐
│    Model     │   │      Template      │   │       View       │
│   (Data)     │   │   (Presentation)   │   │   (Behaviour)    │
│              │   │                    │   │                  │
│ Model classes│   │  Generates HTML    │   │ Python function  │
│ Mapped to DB │   │                    │   │ Mapped to URL    │
└──────────────┘   └────────────────────┘   └──────────────────┘
```

### Django Templates

- Templates are **text files** (usually HTML) with placeholders and logic  
- `{{ var }}` renders the value of `var` from the context  
- `{% tag %}` runs logic: loops, conditionals, inheritance, etc.  
- Place template files in `<app>/templates/<app>/` directory

```html
<!-- website/templates/website/welcome.html -->
<html>
<head><title>{{ name }}'s Page</title></head>
<body>
    <h1>Hi I'm {{ name }}!</h1>
    I'm {{ age }} years old.
</body>
</html>
```

### Calling a Template from a View

```python
# website/views.py
from django.shortcuts import render

def home(request):
    # render(request, template_path, context_dict)
    return render(request, "website/welcome.html", {'name': 'Bob', 'age': 35})
```

**Notes on `render()`:**
- First argument: the `request` object  
- Second: path to the template file (relative to any `templates/` folder)  
- Third: a `dict` with data to pass into the template  
- Don't forget the `return` keyword!

### Retrieving Model Data in a View

```python
# Model classes expose a .objects manager
from meetings.models import Meeting

# Get all meetings
meetings = Meeting.objects.all()

# Count
num_meetings = Meeting.objects.count()

# Get a specific object by primary key
meeting = Meeting.objects.get(pk=5)
```

### URL Parameters and `get_object_or_404`

Use `<int:id>` in the URL pattern to capture a number and pass it to the view.  
If the object doesn't exist, return a proper 404 response.

```python
# meetings/views.py
from django.shortcuts import render, get_object_or_404
from .models import Meeting

def detail(request, id):
    meeting = get_object_or_404(Meeting, pk=id)
    return render(request, "meetings/detail.html", {'meeting': meeting})
```

```python
# meetings/urls.py  — URL with an integer parameter
from django.urls import path
from . import views

urlpatterns = [
    path('<int:id>', views.detail),   # e.g. /meetings/1  → detail(request, id=1)
]
```

### Full Request → Response Diagram (MTV)

```
Browser                              Server
┌──────────────────┐                 ┌────────────────────────────────────┐
│ /meetings/1      │── HTTP GET ────►│ urls.py: path('meetings/<int:id>') │
│                  │                 │         → detail view              │
│                  │                 │                 │                  │
│  Meeting detail  │◄────────────────│ view calls get_object_or_404()     │
│  page rendered   │                 │ fetches from DB, renders template  │
└──────────────────┘                 └────────────────────────────────────┘
```

### Summary — Module 5
- Understood the Model–Template–View architecture  
- Created HTML templates with `{{ variable }}` placeholders  
- Used `render()` to call a template from a view  
- Captured URL parameters with `<int:id>`  
- Returned 404 responses using `get_object_or_404()`

---
## 5. URLs and Link Building

### App-Level URL Files

Instead of putting all URL patterns in the project's `urls.py`, each app manages its own URLs. The project then delegates with `include()`.

```python
# meetings/urls.py  — app-level URL config
from django.urls import path
from .views import detail

urlpatterns = [
    path('<int:id>', detail),
]
```

```python
# meeting_planner/urls.py  — project-level router
from django.contrib import admin
from django.urls import include, path
from website.views import welcome

urlpatterns = [
    path('admin/', admin.site.urls),
    path('', welcome, name='welcome'),
    path('meetings/', include('meetings.urls')),   # delegate to app URLs
]
```

### Named URLs

Give URLs a `name` so templates can refer to them symbolically — if you ever change the URL path, the links update automatically.

```python
# meetings/urls.py  — named URL pattern
from django.urls import path
from . import views

urlpatterns = [
    path('<int:id>', views.detail, name='detail'),
]
```

### URL Namespaces (Best Practice)

Add `app_name` to the app's `urls.py` to namespace all its URL names. This prevents name clashes when two apps both have a URL named `detail`.

```python
# meetings/urls.py — with namespace
from django.urls import path
from . import views

app_name = 'meetings'   # ← declare the namespace

urlpatterns = [
    path('',           views.list,   name='list'),
    path('<int:id>',   views.detail, name='detail'),
    path('new/',       views.new,    name='new'),
    path('<int:id>/edit/',   views.edit,   name='edit'),
    path('<int:id>/delete/', views.delete, name='delete'),
]
```

```python
# In templates, prefix the name with the namespace:
# {% url 'meetings:detail' meeting.id %}  ← namespaced
# {% url 'detail' meeting.id %}           ← without namespace (fragile)
```

```python
# In Python code (views, redirects):
from django.urls import reverse
reverse('meetings:detail', args=[meeting.id])
```

```html
<ul>
  {% for meeting in meetings %}
    <li>
      <a href="{% url 'meetings:detail' meeting.id %}">{{ meeting.title }}</a>
    </li>
  {% endfor %}
</ul>
```

### Template `{% for %}` Loop

Iterate over any iterable (QuerySet, list, etc.) in a template:

```html
<ul>
  {% for m in meetings %}
    <li>{{ m.id }}: {{ m.title }}</li>
  {% empty %}
    <li>No meetings yet.</li>
  {% endfor %}
</ul>
```

> `{% empty %}` renders when the list is empty — always add it to avoid a blank `<ul>`.

### Summary — Module 6
- Moved URL patterns into app-level `urls.py` files  
- Used `include()` to delegate from project URLs to app URLs  
- Named URL patterns for maintainable links  
- Generated links in templates with `{% url 'name' arg %}`  
- Iterated over data with `{% for %}` loops

---
## 6. Templates, Styling and Static Content

### Template Inheritance

Avoids repeating HTML boilerplate across every template.  
Define a base template once; child templates override specific **blocks**.

```html
<!-- templates/base.html -->
<!DOCTYPE html>
<html>
<head>
    <title>Meeting Planner</title>
    {% load static %}
    <link rel="stylesheet" href="{% static 'website/style.css' %}">
</head>
<body>
    <nav><!-- shared navigation --></nav>

    {% block content %}
    <!-- child templates replace this block -->
    {% endblock %}
</body>
</html>
```

```html
<!-- meetings/templates/meetings/list.html -->
{% extends "base.html" %}

{% block content %}
    <h1>Meetings</h1>
    <ul>
        {% for meeting in meetings %}
            <li><a href="{% url 'meetings:detail' meeting.id %}">{{ meeting.title }}</a></li>
        {% empty %}
            <li>No meetings scheduled.</li>
        {% endfor %}
    </ul>
{% endblock %}
```

### Template Inheritance Rules

| Tag | Purpose |
|---|---|
| `{% extends "base.html" %}` | Must be the **first** tag in the child template |
| `{% block name %}` | Declares a replaceable region |
| `{% endblock %}` | Closes the block |
| `{{ block.super }}` | Renders parent block content inside child block |

### Static Files (CSS, JS, Images)

Static files live in `<app>/static/<app>/` — the doubled path prevents naming conflicts.

```html
{% load static %}

<!-- CSS -->
<link rel="stylesheet" href="{% static 'website/style.css' %}">

<!-- Image -->
<img src="{% static 'website/calendar.png' %}" alt="Calendar">
```

### Static Files Directory Structure

```
website/
└── static/
    └── website/        ← app namespace prevents filename collisions
        ├── style.css
        ├── calendar.png
        └── meeting.png
```

### Template Filters

Filters transform a variable's value using `{{ value|filter }}` or `{{ value|filter:"arg" }}`.

```html
{{ meeting.title|upper }}              → TEAM STANDUP
{{ meeting.title|lower }}              → team standup
{{ meeting.title|truncatechars:20 }}   → Team Standup in...
{{ meeting.date|date:"D, d M Y" }}     → Mon, 15 Oct 2026
{{ meeting.date|date:"SHORT_DATE_FORMAT" }}  → 10/15/2026
{{ meeting.location|default:"TBD" }}   → TBD  (if location is empty)
{{ meetings|length }}                  → 5
{{ meeting.title|capfirst }}           → Team standup
```

Common filters:

| Filter | Purpose |
|---|---|
| `\|upper` / `\|lower` | Case conversion |
| `\|title` | Title-case every word |
| `\|truncatechars:N` | Truncate to N characters |
| `\|date:"format"` | Format a date/datetime |
| `\|default:"val"` | Fallback when value is falsy |
| `\|length` | Length of a string or list |
| `\|linebreaks` | Convert `\n` to `<br>` / `<p>` |
| `\|safe` | Mark HTML as safe (skip escaping) — use carefully |
| `\|escape` | Force HTML escaping (the default) |

### Summary — Module 6
- Created a base template with `{% block %}` regions  
- Extended it in child templates with `{% extends %}`  
- Used `{% empty %}` in `{% for %}` loops for empty-list handling  
- Loaded and referenced static files with `{% load static %}` and `{% static 'path' %}`  
- Transformed output with template filters (`\|date`, `\|truncatechars`, `\|default`, etc.)

---
## 7. Adding User Interaction with ModelForms

### ModelForms

A `ModelForm` automatically generates an HTML form from a model class. Django handles field types, labels, and validation for you.

```python
# meetings/views.py
from django.forms import modelform_factory
from django.shortcuts import render, redirect
from .models import Meeting

# Create a form class for the Meeting model
# exclude=[] means include all fields
MeetingForm = modelform_factory(Meeting, exclude=[])

def new(request):
    form = MeetingForm()
    return render(request, "meetings/new.html", {'form': form})
```

### Rendering a Form in a Template

```html
<!-- meetings/templates/meetings/new.html -->
{% extends "base.html" %}

{% block content %}
<h1>New Meeting</h1>
<form method="post">
    {% csrf_token %}
    {{ form.as_p }}
    <button type="submit">Save</button>
</form>
{% endblock %}
```

> Always put `{% csrf_token %}` **before** the form fields — it must be inside the `<form>` tag.

### CSRF Protection

**Cross-Site Request Forgery** protection is built into Django.  
Every `POST` form **must** include `{% csrf_token %}` — Django will reject requests without it.

### The GET/POST Pattern

The same view handles both displaying the empty form (GET) and processing the submitted data (POST):

```python
# meetings/views.py — complete new meeting view
from django.forms import modelform_factory
from django.shortcuts import render, redirect
from .models import Meeting

MeetingForm = modelform_factory(Meeting, exclude=[])

def new(request):
    if request.method == "POST":
        # Form was submitted — bind POST data to the form
        form = MeetingForm(request.POST)
        if form.is_valid():
            form.save()              # saves the new Meeting to the database
            return redirect('welcome')   # redirect after successful save
        # If invalid, fall through and re-render with error messages
    else:
        # GET request — show an empty form
        form = MeetingForm()

    return render(request, "meetings/new.html", {'form': form})
```

### GET/POST Flow Diagram

```
User visits /meetings/new/
        │
        ▼  GET
  View creates empty MeetingForm
  Renders form template
        │
        ▼  User fills form and clicks Save (POST)
  form = MeetingForm(request.POST)
        │
   form.is_valid()?
        ├── YES → form.save() → redirect('welcome')
        └── NO  → re-render template with form (errors shown)
```

### Summary — Module 8
- Created a `ModelForm` from a model using `modelform_factory()`  
- Rendered it in a template with `{{ form }}` and `{% csrf_token %}`  
- Handled GET (display) and POST (validate + save) in one view  
- Used `form.is_valid()` and `form.save()`  
- Redirected after successful save with `redirect()`

---
## 8. Editing Meetings and Authentication

### Editing an Existing Object

Pass the existing instance to `modelform_factory` so the form is pre-populated.

```python
# meetings/views.py — edit view
from django.shortcuts import render, redirect, get_object_or_404
from django.forms import modelform_factory
from .models import Meeting

MeetingForm = modelform_factory(Meeting, exclude=[])

def edit(request, id):
    meeting = get_object_or_404(Meeting, pk=id)

    if request.method == "POST":
        form = MeetingForm(request.POST, instance=meeting)   # bind to existing object
        if form.is_valid():
            form.save()
            return redirect('detail', id=meeting.id)
    else:
        form = MeetingForm(instance=meeting)   # pre-populate form

    return render(request, "meetings/edit.html", {'form': form})
```

### Deleting an Object

```python
# meetings/views.py — delete view (only accepts POST for safety)
from django.shortcuts import redirect, get_object_or_404
from .models import Meeting

def delete(request, id):
    meeting = get_object_or_404(Meeting, pk=id)
    if request.method == "POST":
        meeting.delete()
        return redirect('welcome')
    return redirect('detail', id=id)
```

### ForeignKey — Linking a Meeting to a User

```python
# meetings/models.py — ForeignKey to Django's built-in User model
from django.db import models
from django.contrib.auth.models import User

class Meeting(models.Model):
    title      = models.CharField(max_length=200)
    date       = models.DateField()
    start_time = models.TimeField()
    end_time   = models.TimeField()
    location   = models.CharField(max_length=200)
    attendees  = models.ManyToManyField(User, blank=True)
    # Or a single organizer:
    # organizer = models.ForeignKey(User, on_delete=models.CASCADE)
```

### QuerySets and Laziness

A **QuerySet** is a lazy database query — it is only evaluated (SQL executed) when the data is actually accessed.

```python
from meetings.models import Meeting

# No SQL is executed yet:
meetings = Meeting.objects.all()

# SQL is executed here (when iterating/rendering):
for m in meetings:
    print(m.title)

# Filtering — still lazy
my_meetings = Meeting.objects.filter(organizer=request.user)

# Ordering
sorted_meetings = Meeting.objects.order_by('date')
```

### Authentication: Login and Logout

Django ships with fully-built login/logout views. Wire them up in your URL config and add two settings so Django knows where to redirect after login/logout:

```python
# meeting_planner/settings.py
LOGIN_REDIRECT_URL  = '/'   # redirect here after successful login
LOGOUT_REDIRECT_URL = '/'   # redirect here after logout
```

The built-in `LoginView` looks for a template at `registration/login.html`. Create it:

```html
<!-- templates/registration/login.html -->
{% extends "base.html" %}

{% block content %}
<h1>Log in</h1>
<form method="post">
    {% csrf_token %}
    {{ form.as_p }}
    <button type="submit">Log in</button>
</form>
{% endblock %}
```

```python
# meeting_planner/urls.py — add auth URLs
from django.contrib import admin
from django.urls import include, path
from django.contrib.auth import views as auth_views
from website.views import welcome

urlpatterns = [
    path('admin/', admin.site.urls),
    path('', welcome, name='welcome'),
    path('meetings/', include('meetings.urls')),
    path('accounts/login/', auth_views.LoginView.as_view(), name='login'),
    path('accounts/logout/', auth_views.LogoutView.as_view(), name='logout'),
]
```

### Protecting Views with `@login_required`

```python
# meetings/views.py — restrict access to logged-in users
from django.contrib.auth.decorators import login_required
from django.shortcuts import render, redirect, get_object_or_404
from django.forms import modelform_factory
from .models import Meeting

MeetingForm = modelform_factory(Meeting, exclude=[])

@login_required
def new(request):
    if request.method == "POST":
        form = MeetingForm(request.POST)
        if form.is_valid():
            form.save()
            return redirect('welcome')
    else:
        form = MeetingForm()
    return render(request, "meetings/new.html", {'form': form})

# Unauthenticated users are redirected to settings.LOGIN_URL (default: /accounts/login/)
```

### Reacting to Login Status in Templates

```html
{% if user.is_authenticated %}
    <p>Welcome, {{ user.username }}!</p>
    <a href="{% url 'logout' %}">Log out</a>
    <a href="{% url 'meetings:new' %}">Add Meeting</a>
{% else %}
    <a href="{% url 'login' %}">Log in</a>
{% endif %}
```

### Advanced Admin Configuration

```python
# meetings/admin.py — customise the admin list display
from django.contrib import admin
from .models import Meeting

@admin.register(Meeting)
class MeetingAdmin(admin.ModelAdmin):
    list_display = ('title', 'date', 'start_time', 'location')
    list_filter  = ('date',)
    search_fields = ('title', 'location')
```

### Summary — Module 9
- Built edit and delete views for existing objects  
- Used `ForeignKey` / `ManyToManyField` to link meetings to users  
- Understood QuerySet laziness  
- Wired up Django's built-in login/logout views  
- Protected views with the `@login_required` decorator  
- Checked login status in templates with `{% if user.is_authenticated %}`  
- Customised the Admin interface with `ModelAdmin`

---
## Quick-Reference Cheatsheet

### Essential `manage.py` Commands

| Command | Purpose |
|---|---|
| `runserver` | Start the development server |
| `startapp <name>` | Create a new Django app |
| `shell` | Interactive Python shell with Django loaded |
| `makemigrations` | Generate migration scripts from model changes |
| `showmigrations` | List migrations and their applied status |
| `sqlmigrate <app> <num>` | Preview the SQL for a migration |
| `migrate` | Apply pending migrations to the database |
| `dbshell` | Open an interactive shell for the configured database |
| `createsuperuser` | Create an admin user |
| `collectstatic` | Gather static files into `STATIC_ROOT` (production) |

### Template Tag Reference

| Tag / Syntax | Purpose |
|---|---|
| `{{ variable }}` | Render a variable from the context |
| `{{ obj.attribute }}` | Access an attribute / field |
| `{% for item in list %}` | Loop |
| `{% empty %}` | Fallback inside `{% for %}` when list is empty |
| `{% if condition %}` | Conditional |
| `{% elif condition %}` / `{% else %}` | Branches |
| `{% url 'app:name' arg %}` | Reverse a named URL |
| `{% extends "base.html" %}` | Inherit from a base template |
| `{% block name %}` | Define a replaceable block |
| `{{ block.super }}` | Include parent block content |
| `{% load static %}` | Load the static files tag library |
| `{% static 'path/to/file' %}` | Generate the URL for a static file |
| `{% csrf_token %}` | Add CSRF protection to a POST form |
| `{% include "snippet.html" %}` | Embed another template |
| `{% with var=value %}` | Assign a template variable |
| `{% comment %}` | Comment block (not rendered) |

### Common Template Filters

| Filter | Example | Output |
|---|---|---|
| `\|upper` | `{{ title\|upper }}` | `TEAM STANDUP` |
| `\|lower` | `{{ title\|lower }}` | `team standup` |
| `\|truncatechars:N` | `{{ title\|truncatechars:10 }}` | `Team Stan…` |
| `\|date:"D d M Y"` | `{{ date\|date:"D d M Y" }}` | `Mon 15 Oct 2026` |
| `\|default:"val"` | `{{ location\|default:"TBD" }}` | `TBD` |
| `\|length` | `{{ meetings\|length }}` | `5` |
| `\|linebreaks` | `{{ notes\|linebreaks }}` | Converts `\n` to `<p>` |
| `\|safe` | `{{ html\|safe }}` | Bypasses auto-escaping |

### Common Model Field Types

| Field | Use for |
|---|---|
| `CharField(max_length=N)` | Short text (required `max_length`) |
| `TextField()` | Long text |
| `IntegerField()` | Integer numbers |
| `FloatField()` | Floating-point numbers |
| `DecimalField(max_digits, decimal_places)` | Precise decimals (money) |
| `DateField()` | Dates |
| `TimeField()` | Times |
| `DateTimeField()` | Date + time |
| `BooleanField()` | True/False |
| `EmailField()` | Email address (validates format) |
| `URLField()` | URL (validates format) |
| `ImageField()` | Image upload (requires Pillow) |
| `FileField()` | File upload |
| `ForeignKey(Model, on_delete=...)` | Many-to-one relationship |
| `ManyToManyField(Model)` | Many-to-many relationship |
| `OneToOneField(Model, on_delete=...)` | One-to-one relationship |

### Useful Field Options

| Option | Meaning |
|---|---|
| `blank=True` | Allow empty value in forms |
| `null=True` | Allow `NULL` in the database |
| `default=value` | Default value |
| `unique=True` | Enforce uniqueness at DB level |
| `db_index=True` | Add a database index |
| `verbose_name="label"` | Human-readable label in forms/admin |

### QuerySet API Cheatsheet

```python
from meetings.models import Meeting

# Retrieve all
Meeting.objects.all()

# Filter
Meeting.objects.filter(location='Berlin')
Meeting.objects.filter(date__year=2024)

# Exclude
Meeting.objects.exclude(location='Online')

# Order
Meeting.objects.order_by('date')        # ascending
Meeting.objects.order_by('-date')       # descending

# Get a single object (raises exception if not found)
Meeting.objects.get(pk=1)

# Get or 404 (use in views)
from django.shortcuts import get_object_or_404
meeting = get_object_or_404(Meeting, pk=1)

# Count
Meeting.objects.count()

# Chain filters
Meeting.objects.filter(location='Berlin').order_by('date')
```

---
## Complete Application Overview

The course builds a **Meeting Planner** app with this full structure:

```
meeting_planner/                ← Django project
├── manage.py
├── meeting_planner/
│   ├── settings.py
│   └── urls.py                 ← includes website + meetings URLs
├── website/                    ← app: home page
│   ├── views.py                 (welcome view)
│   ├── static/website/
│   │   ├── style.css
│   │   └── calendar.png
│   └── templates/website/
│       └── welcome.html
└── meetings/                   ← app: meetings CRUD
    ├── models.py                (Meeting, Room models)
    ├── views.py                 (list, detail, new, edit, delete)
    ├── urls.py
    ├── admin.py
    └── templates/meetings/
        ├── list.html
        ├── detail.html
        ├── new.html
        └── edit.html
```

### URLs Summary

| URL | View | Name |
|---|---|---|
| `/` | `website.views.welcome` | `welcome` |
| `/meetings/` | `meetings.views.list` | `list` |
| `/meetings/<id>` | `meetings.views.detail` | `detail` |
| `/meetings/new/` | `meetings.views.new` | `new` |
| `/meetings/<id>/edit/` | `meetings.views.edit` | `edit` |
| `/meetings/<id>/delete/` | `meetings.views.delete` | `delete` |
| `/accounts/login/` | `auth_views.LoginView` | `login` |
| `/accounts/logout/` | `auth_views.LogoutView` | `logout` |
| `/admin/` | Django admin | — |